<a href="https://colab.research.google.com/github/sherf1207/AI-Projects/blob/main/hybrid%20RNN%2C%20LSTM%2C%20GRU%20Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This project compared RNN, LSTM, and GRU models for predicting the direction of Netflix (NFLX) stock prices using time-series data. After proper preprocessing and sequence creation, GRU achieved the best performance, followed by LSTM and RNN. Overall accuracy was moderate due to the noisy and unpredictable nature of financial data. The project highlights the importance of model selection and data preparation in time-series prediction.


In [71]:
import numpy as np
import pandas as pd
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Dense, SimpleRNN, LSTM, GRU

In [72]:
df= pd.read_csv('/content/NFLX.csv')
df

,Date,Open,High,Low,Close,Adj Close,Volume
0,2018-02-05,262.000000,267.899994,250.029999,254.259995,254.259995,11896100
1,2018-02-06,247.699997,266.700012,245.000000,265.720001,265.720001,12595800
2,2018-02-07,266.579987,272.450012,264.329987,264.559998,264.559998,8981500
3,2018-02-08,267.079987,267.619995,250.000000,250.100006,250.100006,9306700
4,2018-02-09,253.850006,255.800003,236.110001,249.470001,249.470001,16906900
...,...,...,...,...,...,...,...
1004,2022-01-31,401.970001,427.700012,398.200012,427.140015,427.140015,20047500
1005,2022-02-01,432.959991,458.480011,425.540009,457.130005,457.130005,22542300
1006,2022-02-02,448.250000,451.980011,426.480011,429.480011,429.480011,14346000
1007,2022-02-03,421.440002,429.260010,404.279999,405.600006,405.600006,9905200


In [73]:
#Convert Date column and sort it just in case
df['Date']= pd.to_datetime(df['Date'])
df= df.sort_values('Date')

In [74]:
features= ['Open', 'High', 'Low', 'Close', 'Volume']
#Removing Date column to scale data
data= df[features].values

In [75]:
#Scaling the data
scaler= MinMaxScaler()
scaledData= scaler.fit_transform(data)

In [76]:
#Creating sequence
sequence_length = 7
x = []
y = []

close_prices = df['Close'].values

for i in range(len(scaledData) - sequence_length):
    x.append(scaledData[i:i+sequence_length])

    #check if the prices went up or down
    if close_prices[i + sequence_length] > close_prices[i + sequence_length - 1]:
        y.append(1)
    else:
        y.append(0)

In [77]:
x= np.array(x)
y= np.array(y)

#as regular TTS function shuffles the data we're gonna make it manually
split = int(0.8 * len(x))
Xtrain, Xtest = x[:split], x[split:]
Ytrain, Ytest = y[:split], y[split:]



# Models training function


In [78]:
def train_model(model_type, X_train, y_train, X_test, y_test):
    model = Sequential()

    if model_type == 'RNN':
        model.add(SimpleRNN(64,
                            input_shape=(Xtrain.shape[1], Xtrain.shape[2]),
                            activation='tanh'))
    elif model_type == 'LSTM':
        model.add(LSTM(64,
                       input_shape=(Xtrain.shape[1], Xtrain.shape[2]),
                       activation='tanh'))
    elif model_type == 'GRU':
        model.add(GRU(64,
                      input_shape=(Xtrain.shape[1], Xtrain.shape[2]),
                      activation='tanh'))
    else:
        raise ValueError("Invalid model ")

    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)
    loss, accuracy = model.evaluate(X_test, y_test, verbose=0)

    return accuracy

In [79]:
#Running comparison between models
RNN_acc = train_model('RNN', Xtrain, Ytrain, Xtest, Ytest)
LSTM_acc = train_model('LSTM', Xtrain, Ytrain, Xtest, Ytest)
GRU_acc = train_model('GRU', Xtrain, Ytrain, Xtest, Ytest)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [80]:
#Printing results
print(f"RNN Accuracy : {RNN_acc:.4f}")
print(f"LSTM Accuracy: {LSTM_acc:.4f}")
print(f"GRU Accuracy : {GRU_acc:.4f}")

RNN Accuracy : 0.5373
LSTM Accuracy: 0.4726
GRU Accuracy : 0.5622
